# 14.2 word2vec: 단어를 벡터로 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter14_2_word2vec.ipynb)

책 본문: [14.2 word2vec: 단어를 벡터로](https://smhanlab.com/book-ml/kor/ml1/chapter14/2.html)

이 노트북은 14.2절의 내용을 코드로 끝까지 실행합니다:
(1) **skip-gram + negative sampling** 학습을 순수 Python으로 재구현,
(2) 학습된 임베딩으로 **코사인 유사도** 비슷한-단어 찾기,
(3) 8차원 임베딩을 **PCA/SVD로 2D 투영** → 동물/과일 군집 시각화,
(4) `neg_k`, `window` 하이퍼파라미터의 효과 정량 측정,
(5) **CBOW vs Skip-gram** 비교,
(6) "왕−남자+여자=여왕"식의 **의미 유사도 연산** 확인,
(7) 임베딩 **노름(길이)** 통계 — "빈도 ∝ 노름" 현상 확인.

전부 로컬에서 `python`으로 먼저 실행 검증한 뒤 커밋했습니다.

In [1]:
import math, random
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
import os
os.makedirs(IMG, exist_ok=True)
random.seed(42)  # 재현 가능성

## 1. skip-gram + negative sampling 재구현

본문의 `train_skipgram`과 `cos_sim`을 그대로 둔다. 핵심:
- `W_in` = 중심 단어 임베딩(최종 결과물), `W_out` = 주변 단어 벡터(학습용만).
- 매 쌍마다 **정답 1개 + 무작위 가짜 `neg_k`개**를 이진 분류(`label=1/0`)로 학습.
- 그래디언트 계수는 로지스틱 회귀와 같은 `(pred - label)` (본문 "손으로 한 번" 참고).

In [2]:
def train_skipgram(corpus, window=2, dim=8, epochs=50, lr=0.05, neg_k=3, mode="skipgram"):
    vocab = sorted(set(corpus))
    idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)
    W_in  = [[random.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]
    W_out = [[random.uniform(-0.5, 0.5) for _ in range(dim)] for _ in range(V)]

    def sigmoid(z):
        return 1 / (1 + math.exp(-max(-20, min(20, z))))

    pairs = []
    if mode == "skipgram":
        for i, center in enumerate(corpus):
            for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
                if j != i:
                    pairs.append((idx[center], idx[corpus[j]]))
    else:  # cbow: (context, center) — 입력=주변, 출력=중심
        for i, center in enumerate(corpus):
            for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
                if j != i:
                    pairs.append((idx[corpus[j]], idx[center]))
    for _ in range(epochs):
        random.shuffle(pairs)
        for c, o in pairs:
            targets = [(o, 1)] + [(random.randrange(V), 0) for _ in range(neg_k)]
            for t, label in targets:
                z = sum(W_in[c][k] * W_out[t][k] for k in range(dim))
                pred = sigmoid(z)
                grad = (pred - label) * lr
                for k in range(dim):
                    g_in, g_out = W_in[c][k], W_out[t][k]
                    W_in[c][k]  -= grad * g_out
                    W_out[t][k] -= grad * g_in
    return {w: W_in[idx[w]] for w in vocab}

def cos_sim(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(x*x for x in b))
    return dot / (na*nb + 1e-9)

## 2. 비슷한 단어 찾기 (본문 실습)

동물/과일 두 그룹의 작은 말뭉치로 학습 후, "고양이는"과 코사인 유사도가
높은 단어 상위 5개를 본다. 본문의 숫자(동물이다 0.875, 사자는 0.622, …)가
나오는지 확인한다.

In [3]:
corpus = ("고양이는 동물이다 강아지는 동물이다 사자는 동물이다 "
          "고양이는 귀엽다 강아지는 귀엽다 "
          "사과는 과일이다 바나나는 과일이다 포도는 과일이다 "
          "사과는 맛있다 바나나는 맛있다").split()
print("토큰 수:", len(corpus), " 고유어 수:", len(set(corpus)))

random.seed(42)
emb = train_skipgram(corpus, window=2, dim=8, epochs=200, lr=0.1, neg_k=4)

query = "고양이는"
sims = sorted(((w, cos_sim(emb[query], emb[w])) for w in emb if w != query),
              key=lambda kv: -kv[1])
for w, s in sims:
    print(f"{w}: {s:.3f}")

# 본문의 숫자와 일치하는지 확인
top = dict(sims)
assert abs(top["동물이다"] - 0.875) < 0.01
assert abs(top["사과는"] - 0.549) < 0.01
print("\n본문 숫자(0.875 / 0.549)와 일치!")

토큰 수: 20  고유어 수: 10
동물이다: 0.875
사자는: 0.622
귀엽다: 0.581
사과는: 0.549
강아지는: 0.533
과일이다: 0.287
바나나는: 0.282
포도는: 0.169
맛있다: 0.084

본문 숫자(0.875 / 0.549)와 일치!


## 3. 2D 투영: "문맥이 비슷한 단어는 가까이"를 눈으로

8차원 임베딩을 SVD의 상위 2 좌표(PCA와 동일한 선형 투영)로 줄여
산점도로 본다. 동물 관련 단어(고양이는·강아지는·사자는·동물이다·귀엽다)와
과일 관련 단어(사과는·바나나는·포도는·과일이다·맛있다)가 양쪽으로
군집되는지 확인한다.

In [4]:
def svd_2d(M_dict):
    words = list(M_dict)
    M = np.array([M_dict[w] for w in words])
    M = M - M.mean(axis=0)
    u, s, vt = np.linalg.svd(M, full_matrices=False)
    return words, M @ vt[:2].T

words, P2 = svd_2d(emb)
animals = {"고양이는","강아지는","사자는","동물이다","귀엽다"}
fruits  = {"사과는","바나나는","포도는","과일이다","맛있다"}

# data label을 영어로 (Colab CJK 폰트 안전망)
labels = {"고양이는": "cat", "강아지는": "dog", "사자는": "lion", "동물이다": "is-an-animal", "귀엽다": "cute",
          "사과는": "apple", "바나나는": "banana", "포도는": "grape", "과일이다": "is-a-fruit", "맛있다": "tasty"}

fig, ax = plt.subplots(figsize=(7,5.5), dpi=110)
ax.scatter(*np.array(P2).T, s=140, c="lightgray", zorder=2)
for w, (x, y) in zip(words, P2):
    color = "tab:blue" if w in animals else "tab:orange"
    ax.annotate(labels[w], (x, y), color=color, fontsize=11,
                ha="center", va="center", fontweight="bold")
# 그룹 경계(수직선)
ax.axvline(0, color="gray", ls="--", lw=1)
ax.set_title("word2vec embedding 2D projection (top 2 SVD coordinates)", fontsize=13)
ax.set_xlabel("1st principal component"); ax.set_ylabel("2nd principal component")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch14_2_word2vec_2d.svg", bbox_inches="tight")
fig.savefig("/tmp/ch14_2_word2vec_2d.png", bbox_inches="tight")
plt.show()
print("SVG 저장:", IMG + "/ch14_2_word2vec_2d.svg")

# 정량 확인: 두 그룹의 centroid가 서로 다른 쪽에 있는지
pa = np.mean([P2[words.index(w)] for w in animals], axis=0)
pf = np.mean([P2[words.index(w)] for w in fruits], axis=0)
print(f"동물 centroid: ({pa[0]:+.2f}, {pa[1]:+.2f})   과일 centroid: ({pf[0]:+.2f}, {pf[1]:+.2f})")
assert pa[0] < 0 < pf[0], "동물/과일 centroid가 양쪽에 나눠져야 함"
print("centroid가 서로 반대 쪽에 위치 — 군집 구조 확인!")

SVG 저장: /home/smhan/book-ml/kor/src/images/ch14_2_word2vec_2d.svg
동물 centroid: (-1.05, -0.09)   과일 centroid: (+1.05, +0.09)
centroid가 서로 반대 쪽에 위치 — 군집 구조 확인!


## 4. 하이퍼파라미터 스weep: `neg_k`와 `window`

본문 "자주 하는 실수"에서 말한 현상을 정량 확인한다.

**실험 A — `neg_k`의 효과**: `neg_k=0`(정답만 학습) vs `neg_k=4`.
neg_k=0이면 "함께 안 나오는 쌍을 내리는 학습"이 없으므로,
공유 문맥("동물이다")을 통해 연결된 동물 단어들의 벡터가
서로 같은 방향으로 수렴해 유사도가 극단적으로 높아지는 것이 보인다.

**실험 B — `window`의 효과**: "동물–과일 구분력"
(sim(고양이는, 사자는) − sim(고양이는, 바나나는))을 window별로 측정.

In [5]:
# 실험 A: neg_k sweep
print("=== neg_k sweep (epochs=200, window=2) ===")
print(f"{'neg_k':>5}  {'sim(사자는)':>11}  {'sim(바나나는)':>13}  {'gap':>7}")
gaps = {}
for k in [0, 1, 4, 8]:
    random.seed(42)
    e = train_skipgram(corpus, window=2, dim=8, epochs=200, lr=0.1, neg_k=k)
    sa = cos_sim(e["고양이는"], e["사자는"]); sf = cos_sim(e["고양이는"], e["바나나는"])
    gaps[k] = sa - sf
    print(f"{k:>5}  {sa:>+11.3f}  {sf:>+13.3f}  {sa-sf:>+7.3f}")

# 실험 B: window sweep
print("\n=== window sweep (epochs=200, neg_k=4) ===")
for wdw in [1, 2, 4]:
    random.seed(42)
    e = train_skipgram(corpus, window=wdw, dim=8, epochs=200, lr=0.1, neg_k=4)
    gap = cos_sim(e["고양이는"], e["사자는"]) - cos_sim(e["고양이는"], e["바나나는"])
    print(f"window={wdw}: gap = {gap:+.3f}")

=== neg_k sweep (epochs=200, window=2) ===
neg_k     sim(사자는)      sim(바나나는)      gap
    0       +0.928         -0.081   +1.009
    1       +0.460         -0.071   +0.531
    4       +0.622         +0.282   +0.341


    8       +0.692         +0.283   +0.409

=== window sweep (epochs=200, neg_k=4) ===
window=1: gap = +0.303
window=2: gap = +0.341


window=4: gap = +0.040


## 5. CBOW vs Skip-gram

같은 말뭉치·같은 하이퍼파라미터로 두 방식을 학습해
"사과는"의 비슷한 단어 상위 5개를 비교한다.
(희귀단어 정확도 차이 같은 미세한 차이는 이 작은 말뭉치에서는
보이지 않을 수 있다 — 둘 다 "동일한 구조"를 학습하기 때문.
차이가 크게 나려면 희귀단어가 섞인 큰 말뭉치가 필요하다.)

In [6]:
random.seed(42)
e_sg = train_skipgram(corpus, window=2, dim=8, epochs=200, lr=0.1, neg_k=4, mode="skipgram")
random.seed(42)
e_cb = train_skipgram(corpus, window=2, dim=8, epochs=200, lr=0.1, neg_k=4, mode="cbow")

for name, e in [("Skip-gram", e_sg), ("CBOW", e_cb)]:
    q = "사과는"
    s = sorted(((w, cos_sim(e[q], e[w])) for w in e if w != q), key=lambda kv: -kv[1])
    top5 = ", ".join(f"{w}({v:.2f})" for w, v in s[:5])
    print(f"{name}: {top5}")

Skip-gram: 과일이다(0.72), 고양이는(0.55), 맛있다(0.54), 바나나는(0.51), 귀엽다(0.48)
CBOW: 과일이다(0.71), 맛있다(0.51), 고양이는(0.49), 포도는(0.48), 귀엽다(0.40)


## 6. 의미 유사도 연산: "축 전환"

본문 "왕 − 남자 + 여자 = 여왕"의 구조를 작은 말뭉치 버전으로 확인:
`vec(고양이는) − vec(귀엽다) + vec(맛있다)`와 가장 비슷한 단어를 찾는다.
"동물-속성 → 과일-속성" 축 전환이 일어나 "사과" 쪽이 1위가 되어야 한다.

In [7]:
target = [a - b + c for a, b, c in zip(emb["고양이는"], emb["귀엽다"], emb["맛있다"])]
analogy = sorted(((w, cos_sim(target, emb[w])) for w in emb
                  if w not in {"고양이는", "귀엽다", "맛있다"}),
                 key=lambda kv: -kv[1])[:3]
print("analogy 상위 3:", [(w, round(s,3)) for w, s in analogy])
assert analogy[0][0] == "사과는", "축 전환 결과 1위가 '사과는'여야 함"
print("1위가 '사과는' — 축 전환 구조 확인!")

analogy 상위 3: [('사과는', 0.571), ('바나나는', 0.478), ('과일이다', 0.456)]
1위가 '사과는' — 축 전환 구조 확인!


## 7. 임베딩 노름: "빈도 ∝ 노름"

코사인 유사도는 노름을 무시하고(각도만 측정) **유사도 연산**은
노름 차이에 영향을 받는다는 본문 "자주 하는 실수"의 근거:
말뭉치 빈도와 임베딩 노름을 나란히 비교한다.

In [8]:
freq = {w: corpus.count(w) for w in emb}
print(f"{'단어':>8}  {'빈도':>4}  {'노름':>6}")
for w in sorted(emb, key=lambda w: -math.sqrt(sum(x*x for x in emb[w]))):
    print(f"{w:>8}  {freq[w]:>4}  {math.sqrt(sum(x*x for x in emb[w])):>6.2f}")
print("\n(참고) 노름은 학습의 무작위성 때문에 빈도와 완전히 비례하지는 않지만,")
print("빈도가 높은 단어일수록 벡터가 더 길어지는 경향이 보인다.")

      단어    빈도      노름
     사자는     1    2.88
     포도는     1    2.59
    바나나는     2    2.46
     맛있다     2    2.40
    고양이는     2    2.39
    강아지는     2    2.19
    과일이다     3    2.04
    동물이다     3    2.02
     사과는     2    2.00
     귀엽다     2    1.93

(참고) 노름은 학습의 무작위성 때문에 빈도와 완전히 비례하지는 않지만,
빈도가 높은 단어일수록 벡터가 더 길어지는 경향이 보인다.


## 정리

- **skip-gram**은 "중심 단어 → 주변 단어" 이진 분류(정답 1 + 가짜 `neg_k`)로
  학습하며, 그 **입력 가중치 행**이 곧 단어 임베딩.
- 학습된 임베딩은 **문맥이 비슷한 단어끼리 벡터 공간에서 가까워지고**,
  2D 투영에서도 동물/과일 두 군집이 분리된다.
- `neg_k`는 "안 나오는 쌍을 내리는 힘"을 결정 — 0이면 공유 문맥을
  경유한 단어들이 극단적으로 유사해진다.
- **유사도 연산**(a − b + c)은 의미 축의 병렬 이동을 이용하며,
  축이 일관된 스케일로 학습되면 "사과"처럼 1위가 나온다.
- 임베딩 **노름**은 빈도 효과를 담는다 — 비슷한 단어 찾기는 코사인(각도),
  analogy는 노름 차이를 함께 고려한다.

(14.3절에서는 이 `train_skipgram`을 그래프의 **무작위 걷기** 시퀀스에
적용해 Node2Vec를 만든다 — "문장 대신 노드 경로"로 바꾸는 것뿐.)